# 🎙️ XFarmaCare — Pipeline de Transcripción y Análisis de Sentimientos
## De audio del call center a variable predictiva

**Propósito:** Tomar los audios crudos de las llamadas del call center,
transcribirlos con Whisper (OpenAI) y clasificar el sentimiento de cada
interacción con un modelo BERT multilingüe.

**El output se convierte en input para el modelo de churn.**
Las columnas de sentimiento (puntaje negativo, estrellas, etc.) son
features predictivas que alimentan directamente al notebook `01`.

**Pipeline:**
```
Audio (.wav/.mp3) → Whisper (transcripción) → BERT (sentimiento)
    → CSV con scores → Merge con fact_interacciones_callcenter
        → Feature engineering → Modelo de churn
```

**Requisitos:**
```bash
pip install openai-whisper transformers torch pandas
```

---
**Nota:** Este notebook se ejecuta cada vez que hay un lote nuevo de audios.
En producción, se puede automatizar como cronjob o trigger por carpeta.

In [ ]:
# ══════════════════════════════════════════════════════
# CONFIGURACIÓN
# ══════════════════════════════════════════════════════
import os
import glob
import pandas as pd
from datetime import datetime

# --- Rutas ---
# Cambiar AUDIO_DIR a la carpeta donde están los audios del call center
AUDIO_DIR = "audio/"                          # ← Carpeta con los .wav/.mp3
OUTPUT_CSV = "outputs/resultados_sentimiento_callcenter.csv"
CALLCENTER_CSV = "data/fact_interacciones_callcenter.csv"

# Extensiones de audio que buscará
EXTENSIONES = ["*.wav", "*.mp3", "*.m4a", "*.ogg", "*.flac", "*.webm"]

# Modelo Whisper: tiny | base | small | medium | large
# "small" da buen balance entre velocidad y calidad en español
WHISPER_MODEL = "small"

# Crear carpetas si no existen
os.makedirs("outputs", exist_ok=True)
os.makedirs(AUDIO_DIR, exist_ok=True)

print(f"Carpeta de audio: {os.path.abspath(AUDIO_DIR)}")
print(f"Output CSV:       {os.path.abspath(OUTPUT_CSV)}")

## 1. Cargar los modelos
Whisper se encarga de convertir el audio a texto.
El modelo BERT multilingüe clasifica ese texto en una escala de 1 a 5 estrellas,
que luego agrupamos en POSITIVO / NEUTRO / NEGATIVO.

In [ ]:
# ══════════════════════════════════════════════════════
# CARGAR MODELOS (esto tarda la primera vez que descarga)
# ══════════════════════════════════════════════════════
import whisper
from transformers import pipeline

print("Cargando Whisper (modelo de transcripción)...")
modelo_whisper = whisper.load_model(WHISPER_MODEL)
print(f"  → Whisper '{WHISPER_MODEL}' cargado.")

print("Cargando BERT multilingüe (análisis de sentimiento)...")
# Este modelo fue entrenado en reseñas en múltiples idiomas,
# incluyendo español. Devuelve estrellas del 1 al 5.
analizador = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    top_k=None   # Devuelve probabilidad de cada estrella
)
print("  → BERT sentiment cargado.")

## 2. Buscar y procesar los audios
Recorre la carpeta de audio, transcribe cada archivo y clasifica el sentimiento.
Si un audio falla, lo marca y sigue con el siguiente.

In [ ]:
# ══════════════════════════════════════════════════════
# BUSCAR ARCHIVOS DE AUDIO
# ══════════════════════════════════════════════════════
archivos = []
for ext in EXTENSIONES:
    archivos.extend(glob.glob(os.path.join(AUDIO_DIR, ext)))

if not archivos:
    print(f"⚠️  No se encontraron audios en: {os.path.abspath(AUDIO_DIR)}")
    print(f"   Coloca los archivos .wav/.mp3 en esa carpeta y vuelve a ejecutar.")
    print(f"   El pipeline generará el CSV cuando haya audios disponibles.")
else:
    print(f"✅ Se encontraron {len(archivos)} archivos de audio.")

In [ ]:
# ══════════════════════════════════════════════════════
# PROCESAR CADA ARCHIVO
# ══════════════════════════════════════════════════════
resultados = []

for i, ruta in enumerate(archivos, 1):
    nombre = os.path.basename(ruta)
    print(f"[{i}/{len(archivos)}] Procesando: {nombre}")
    
    # --- PASO 1: Transcribir con Whisper ---
    try:
        resultado_whisper = modelo_whisper.transcribe(ruta, language="es")
        texto = resultado_whisper["text"].strip()
    except Exception as e:
        print(f"  ⚠ Error en transcripción: {e}")
        texto = ""
    
    # Si no se pudo extraer texto, registrar y seguir
    if not texto:
        print("  ⚠ Sin texto, se omite el análisis de sentimiento.")
        resultados.append({
            "archivo": nombre,
            "transcripcion": "",
            "sentimiento": "SIN_TEXTO",
            "puntaje_positivo": 0,
            "puntaje_negativo": 0,
            "puntaje_neutro": 0,
            "estrellas": 0,
            "fecha_proceso": datetime.now().isoformat()
        })
        continue
    
    # --- PASO 2: Analizar sentimiento con BERT ---
    # El modelo tiene un límite de 512 tokens, así que cortamos
    texto_corto = texto[:512]
    
    try:
        scores = analizador(texto_corto)[0]  # Lista de {label, score}
        
        # Convertir a diccionario: {"1 star": 0.05, "2 stars": 0.10, ...}
        score_dict = {s["label"]: round(s["score"], 4) for s in scores}
        
        # La estrella con mayor probabilidad
        mejor = max(scores, key=lambda x: x["score"])
        estrellas = int(mejor["label"].split()[0])
        
        # Agrupar en positivo / neutro / negativo
        negativo = score_dict.get("1 star", 0) + score_dict.get("2 stars", 0)
        neutro   = score_dict.get("3 stars", 0)
        positivo = score_dict.get("4 stars", 0) + score_dict.get("5 stars", 0)
        
        # Clasificar
        if estrellas >= 4:
            sentimiento = "POSITIVO"
        elif estrellas == 3:
            sentimiento = "NEUTRO"
        else:
            sentimiento = "NEGATIVO"
    
    except Exception as e:
        print(f"  ⚠ Error en análisis: {e}")
        sentimiento, positivo, negativo, neutro, estrellas = "ERROR", 0, 0, 0, 0
    
    print(f"  → {sentimiento} ({estrellas}★) | {texto[:80]}...")
    
    resultados.append({
        "archivo": nombre,
        "transcripcion": texto,
        "sentimiento": sentimiento,
        "puntaje_positivo": round(positivo, 4),
        "puntaje_negativo": round(negativo, 4),
        "puntaje_neutro": round(neutro, 4),
        "estrellas": estrellas,
        "fecha_proceso": datetime.now().isoformat()
    })

print(f"\nProcesamiento completado: {len(resultados)} archivos.")

## 3. Exportar resultados
El CSV resultante tiene una fila por audio con la transcripción,
el sentimiento clasificado y los puntajes de cada categoría.

Este archivo se usa de dos formas:
1. **Directo:** Se une con `fact_interacciones_callcenter.csv` por nombre de archivo
2. **Como referencia:** Las columnas de sentimiento ya están incluidas
   en el fact del call center (generadas por este pipeline)

In [ ]:
# ══════════════════════════════════════════════════════
# EXPORTAR A CSV
# ══════════════════════════════════════════════════════
if resultados:
    df = pd.DataFrame(resultados)
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    
    print(f"\n✅ CSV guardado: {OUTPUT_CSV}")
    print(f"   Total procesados: {len(df)}")
    print(f"   Positivos: {len(df[df['sentimiento']=='POSITIVO'])}")
    print(f"   Neutros:   {len(df[df['sentimiento']=='NEUTRO'])}")
    print(f"   Negativos: {len(df[df['sentimiento']=='NEGATIVO'])}")
    print(f"   Errores:   {len(df[df['sentimiento'].isin(['SIN_TEXTO','ERROR'])])}")
    
    df.head(10)
else:
    print("No hay resultados para exportar.")
    print("Coloca archivos de audio en la carpeta y vuelve a ejecutar.")

## 4. Integrar con el modelo estrella
Aquí se une el output del pipeline con la tabla de interacciones del call center.
Esto permite que las columnas de sentimiento estén disponibles como features
para el modelo de churn.

In [ ]:
# ══════════════════════════════════════════════════════
# MERGE CON FACT_INTERACCIONES_CALLCENTER
# ══════════════════════════════════════════════════════
# Este paso conecta los resultados del pipeline con la tabla
# de interacciones, que ya tiene el customer_id.
#
# La conexión se hace por nombre de archivo de audio.
# En producción, el sistema de call center asigna un ID de
# interacción a cada grabación.

if os.path.exists(OUTPUT_CSV) and os.path.exists(CALLCENTER_CSV):
    sentimientos = pd.read_csv(OUTPUT_CSV)
    callcenter = pd.read_csv(CALLCENTER_CSV)
    
    print(f"Registros de sentimiento: {len(sentimientos)}")
    print(f"Registros de call center: {len(callcenter)}")
    
    # Si el call center ya tiene las columnas de sentimiento
    # (porque se generaron con datos sintéticos), mostramos
    # cómo sería el merge en producción:
    print("\n--- Cómo funciona el merge en producción ---")
    print("callcenter_enriquecido = callcenter.merge(")
    print("    sentimientos[['archivo', 'transcripcion', 'sentimiento',")
    print("                  'puntaje_positivo', 'puntaje_negativo',")
    print("                  'puntaje_neutro', 'estrellas']],")
    print("    left_on='audio_filename',  # Columna en la tabla del call center")
    print("    right_on='archivo',        # Columna del pipeline")
    print("    how='left'")
    print(")")
    print("\nEsto agrega las columnas de sentimiento a cada interacción,")
    print("que luego se agregan a nivel cliente como features para el churn.")
else:
    print("Ejecuta primero el pipeline de audio para generar el CSV.")

## 5. Opcional: Insertar en SQL Server
Si la empresa usa una base de datos relacional, los resultados
se pueden insertar directamente en una tabla.

In [ ]:
# ══════════════════════════════════════════════════════
# OPCIÓN: INSERTAR EN SQL SERVER
# ══════════════════════════════════════════════════════
# Descomentar el bloque para usarlo.
# Ajustar SERVER, DATABASE y DRIVER según el entorno.

'''
from sqlalchemy import create_engine

SERVER   = "tu_servidor"
DATABASE = "tu_base_de_datos"
DRIVER   = "ODBC Driver 17 for SQL Server"

connection_string = (
    f"mssql+pyodbc://{SERVER}/{DATABASE}"
    f"?driver={DRIVER}&trusted_connection=yes"
)
engine = create_engine(connection_string)

# Insertar (replace = borra y recrea; append = agrega filas)
df.to_sql(
    name="resultados_sentimiento_csat",
    con=engine,
    if_exists="replace",
    index=False,
    schema="dbo"
)
print("✅ Datos insertados en SQL Server: dbo.resultados_sentimiento_csat")
'''

# --- SQL para crear la tabla manualmente ---
print("CREATE TABLE dbo.resultados_sentimiento_csat (")
print("    archivo             NVARCHAR(500),")
print("    transcripcion       NVARCHAR(MAX),")
print("    sentimiento         NVARCHAR(20),")
print("    puntaje_positivo    FLOAT,")
print("    puntaje_negativo    FLOAT,")
print("    puntaje_neutro      FLOAT,")
print("    estrellas           INT,")
print("    fecha_proceso       DATETIME2")
print(");")

## ✅ Pipeline completado

**Flujo dentro del ecosistema XFarmaCare:**

```
00_speech_sentiment_pipeline.ipynb (ESTE NOTEBOOK)
    ↓ genera: resultados_sentimiento_callcenter.csv
    ↓
01_modelo_churn_entrenamiento.ipynb
    ↓ usa las columnas de sentimiento como features
    ↓
02_modelo_churn_scoring_diario.ipynb
    ↓ scoring diario con las features de sentimiento actualizadas
```

**Para automatizar como cronjob:**
```bash
jupyter nbconvert --to script 00_speech_sentiment_pipeline.ipynb
# Ejecutar cada vez que haya audios nuevos en la carpeta
0 5 * * * cd /ruta/xfarmacare/notebooks && python 00_speech_sentiment_pipeline.py >> ../logs/sentiment.log 2>&1
```

**Importante:** Este notebook debe correr ANTES que el 01 y el 02,
porque sus resultados alimentan las features de sentimiento.